# Mental Health in Technology-related Jobs

## Imports

Import libraries for data manipulation, numerical operations, reading the ZIP file, and plotting.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt

## Load and inspect the Dataset

The dataset "OSMI Mental Health in Tech Survey 2016" was downloaded from the website Kaggle (https://www.kaggle.com/datasets/osmi/mental-health-in-tech-2016). The file is called archive.zip.

In [ ]:
with zipfile.ZipFile("data/archive.zip") as archive:
    csv_files = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]
    print("CSV files found:", csv_files)
    with archive.open(csv_files[0]) as file:
        df = pd.read_csv(file)

Inspection of the dataset.

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.info(verbose=True, show_counts=True)

Measuring the missing values per column.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(1)
}).sort_values("Missing count", ascending=False)

with pd.option_context("display.max_rows", None):
    display(missing_summary)

### Missing values and self-employment

Check whether missing answers about the employer´s industry correspond to self-employed respondents.

In [ ]:
pd.crosstab(
    df["Are you self-employed?"],
    df["Is your employer primarily a tech company/organization?"],
    dropna=False
)

**Finding:** All 287 self-employed respondents have a missing answer while all 1146 non-self-employed respondents answered. This supports interpreting these missing values as not applicable.

### Missing values and previous employment

Check whether missing answers about previous employers' mental health benefits correspond to respondents without previous employers.

In [ ]:
pd.crosstab(
    df["Do you have previous employers?"],
    df["Have your previous employers provided mental health benefits?"],
    dropna=False
)

**Finding:** All 169 respondents without previous employers have a missing answer, while all 1264 respondents with previous employers answered. This supports interpreting these missing values as not applicable.

### Missing values and tech/IT roles

Check whether responses to the individual tech/IT-role question depend on whether the respondent works at a tech company.

In [ ]:
pd.crosstab(
    df["Is your employer primarily a tech company/organization?"].fillna("Not applicable"),
    df["Is your primary role within your company related to tech/IT?"].fillna("Missing"),
    dropna=False
)

**Finding:** Only respondents at non-tech companies answered the role question: 248 reported a tech/IT-role and 15 did not. Answers are missing for all 883 tech-company respondents and 287 self-employed respondents. Missing answers do not establish these individuals' roles.


**Implementation note:** The fillna() calls label missing values only for this table. The original dataset remains unchanged.

## Exploratory Data Analysis

Showing the 15 most frequent values per column, including missing values.

In [ ]:
for column in df.columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(15))

Checking the age distribution and its extremes for potentially implausible entries.

In [ ]:
age = df["What is your age?"]

print(age.describe())

print("\nFive lowest ages:")
print(age.sort_values().head())

print("\nFive highest ages:")
print(age.sort_values().tail())

**Finding:** The lowest age is 3 and the highest age is 323. This indicates, that there exist wrong entries in the age column.

Checking for different spelling variations of the gender, after temporarily standardizing capitalization and surrounding spaces.

In [ ]:
gender_normalized = (
    df["What is your gender?"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(gender_normalized.value_counts(dropna=False).to_string())

**Finding:** Being male was answered in different  forms, such as "male", "m", and "man". The same applies to female responses. There also exist uninformative responses such as "human" and "none of your busines". Distinct gender identities should not be interpreted as spelling errors. 3 responses are missing.

Checking for the frequency of every age, including rare extremes.

In [ ]:
print(age.value_counts().sort_index().to_string())

**Finding:** Candidates for potentially invalid entries in the age column, such as 3, 99, and 323 all exist only once in the dataset. 3 and 323 are implausible in this context, while 99 is unusual but not necessarily incorrect.

Checking for identical records.

In [ ]:
print("Identical rows beyond the first occurrence:", df.duplicated().sum())

**Finding:** No completely identical rows were found.

Creating a bar chart to show Yes/Maybe/No/Missing values, for comfort discussing mental health with a supervisor.

In [ ]:
supervisor_question = (
    "Would you feel comfortable discussing a mental health disorder "
    "with your direct supervisor(s)?"
)

response_counts = (
    df[supervisor_question]
    .fillna("Missing")
    .value_counts()
    .reindex(["Yes", "Maybe", "No", "Missing"], fill_value=0)
)

ax = response_counts.plot.bar(rot=0, figsize=(7, 4))
ax.set(
    title="Comfort discussing mental health with a direct supervisor",
    xlabel="Survey response",
    ylabel="Number of respondents"
)

plt.tight_layout()
plt.show()

display(response_counts)

**Finding:** 428 respondents answered "Yes", 382 answered "Maybe", and 336 answered "No". 287 respondents did not answer this question at all.

## Data Cleaning

Creating a separate copy to preserve the original data. The cleaning will be performed on this copied data.

In [ ]:
df_clean = df.copy()

Replacing the two clearly implausible ages (3 and 323) with missing values.

In [ ]:
age_column = "What is your age?"
invalid_age = df_clean[age_column].isin([3, 323])

print("Age values marked as missing:", invalid_age.sum())

df_clean.loc[invalid_age, age_column] = np.nan

The two age values (3 and 323) were changed, but no respondents were removed.

Removing surrounding spaces, standardizing capitalization, and combining clear synonyms. Distinct or ambiguous responses that were not mapped were retained after trimming spaces and converting to lowercase. The original gender column was preserved.

In [ ]:
gender_column = "What is your gender?"

gender_standardization = (
    df_clean[gender_column]
    .astype("string")
    .str.strip()
    .str.lower()
)

gender_mapping = {
    "m": "male",
    "man": "male",
    "cis male": "male",
    "male (cis)": "male",
    "cis man": "male",
    "f": "female",
    "woman": "female",
    "cis female": "female",
    "cis-woman": "female",
    "cisgender female": "female",
    "female/woman": "female",
    "nonbinary": "non-binary",
    "enby": "non-binary"
}

df_clean["gender_clean"] = gender_standardization.replace(gender_mapping)

print(df_clean["gender_clean"].value_counts(dropna=False).to_string())

Spaces and capitalization were standardized, clear synonyms were combined. In total, there are 1049 male, 333 female, and 7 non-binary counts as well as 3 missing responses. Other responses need further review and additional mapping.

In [ ]:
additional_mapping = {
    "fem": "female",
    "mail": "male",
    "malr": "male",
    "sex is male": "male",
    "m|": "male",
    "dude": "male",
    "cisdude": "male",
    "male.": "male",
    "male (trans, ftm)": "male",
    "female (props for making this a freeform field, though)": "female",
    "i identify as female.": "female",
    "mtf": "female",
    "transitioned, m2f": "female",
    "transgender woman": "female",
    "i'm a man why didn't you make this a drop down question. you should of asked sex? and i would of answered yes please. seriously how much text can this take?": "male",
    "human": pd.NA,
    "unicorn": pd.NA,
    "none of your business": pd.NA
}

df_clean["gender_clean"] = (
    df_clean["gender_clean"].replace(additional_mapping))

print(df_clean["gender_clean"].value_counts(dropna=False).to_string())

**Finding:** in total, after the final mapping, there are 1058 male, 339 female, 7 non-binary, and 7 missing answers in the gender column.

Since this analysis is aimed at employer HR measures, the clustering analysis will focus on non-self-employed respondents who work at a tech company or have a tech/IT role. This includes 1131 respondents. It excludes self-employed respondents and the 15 respondents reporting non-tech roles in non-tech companies.

In [ ]:
self_employed = df_clean["Are you self-employed?"]
tech_company = df_clean[
    "Is your employer primarily a tech company/organization?"
]
tech_role = df_clean[
    "Is your primary role within your company related to tech/IT?"
]

analysis_mask = (
    self_employed.eq(0)
    & (tech_company.eq(1) | tech_role.eq(1))
)

df_analysis = df_clean.loc[analysis_mask].copy()

print("Respondents retained:", len(df_analysis))
print("Respondents excluded:", len(df_clean) - len(df_analysis))

**Finding:** 1131 respondents are retained, while 302 respondents (287 self-employed and 15 non-tech respondents at non-tech companies) are excluded. df_clean still contains all respondents, but the analysis now covers 1131 respondents.

## Feature Engineering and Selection

The goal is to identify groups of respondents with similar experiences of workplace mental health support and perceived barriers. For this purpose, the features covering workplace support, perceived stigma, and comfort discussing mental health are selected as a starting point.

In [ ]:
clustering_features = [
    "Does your employer provide mental health benefits as part of healthcare coverage?",
    "Do you know the options for mental health care available under your employer-provided coverage?",
    "Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",
    "Does your employer offer resources to learn more about mental health concerns and options for seeking help?",
    "Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?",
    "If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",
    "Do you think that discussing a mental health disorder with your employer would have negative consequences?",
    "Would you feel comfortable discussing a mental health disorder with your coworkers?",
    "Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?",
    "Do you feel that your employer takes mental health as seriously as physical health?"
]

df_features = df_analysis[clustering_features].copy()

for column in df_features.columns:
    print(F"\n--- {column} ---")
    print(df_features[column].value_counts(dropna=False))

There are 133 missing answers for the question asking about knowing covering options. Further, uncertainty responses such as "I don't know" will remain distinct from "No".

Comparing missing coverage-awareness answers with reported benefits.

In [ ]:
benefits_column = clustering_features[0]
awareness_column = clustering_features[1]

display(pd.crosstab(
    df_features[benefits_column],
    df_features[awareness_column].fillna("Missing"),
    dropna=False
))

The 133 missing answers do not follow one clear branch. 72 occur with "No" benefits, 55 with "Not eligible", and 6 with "I don't know". None occur with "Yes". For that reason, missing answers will be preserved separately.

Using one-hot encoding by changing each response category to a binary column. Missing answers will be represented.

In [ ]:
df_features_prepared = df_features.fillna("Missing")

X = pd.get_dummies(
    df_features_prepared,
    drop_first=False,
    dtype=int
)

print("Encoded matrix shape:", X.shape)
print("Remaining missing values:", X.isna().sum().sum())

display(X.head())

One-hot encoding produced 35 binary columns from 10 survey questions and retained all 1131 respondents. Missing answers were encoded as a separate category.

## Clustering and Evaluation

Imports for Clustering.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score

Comparing cluster counts by using inertia and silhouette scores. The K-Means is an initial approach applied to the one-hot representation.

In [ ]:
evaluation_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X)

    evaluation_results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X, labels),
        "smallest_cluster": pd.Series(labels).value_counts().min()
    })

evaluation = pd.DataFrame(evaluation_results)

display(evaluation)

**Finding:** The highest silhouette score is achieved by k=2, which suggests weak separation. The smallest cluster at k=2 contains 429 respondents. Therefore, 2 clusters are a provisional choice.

Examining the 2 cluster solution and comparing response percentages to understand each group.

In [ ]:
baseline_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

baseline_labels = baseline_model.fit_predict(X)

cluster_labels = pd.Series(
    baseline_labels,
    index=X.index,
    name="cluster"
)

print("Cluster sizes:")
print(cluster_labels.value_counts().sort_index())

cluster_profiles = X.groupby(cluster_labels).mean().T * 100

with pd.option_context("display.max_rows", None):
    display(cluster_profiles.round(1))

**Finding:** In the baseline model, cluster 0 contains 429 and cluster 1 contains 702 respondents.
In cluster 0, there is greater perceived support and openness, for example 82.8% feel comfortable discussing mental health with supervisors.
In cluster 1, there are greater barriers and uncertainty, for example only 9.8% feel comfortable discussing it with supervisors and 78.1% do not know whether anonymity is protected.
The silhouette score of 0.121 suggests overlapping groups, therefore, these descriptions remain provisional and reflect respondents' perception.

Checking whether different random seeds produce similar assignments.

In [ ]:
stability_results = []

for seed in [0, 1, 2, 3, 4]:
    comparison_model = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=20
    )

    comparison_labels = comparison_model.fit_predict(X)

    stability_results.append({
        "seed": seed,
        "ARI_vs_baseline": adjusted_rand_score(
            baseline_labels, comparison_labels
        )
    })

display(pd.DataFrame(stability_results))

**Finding:** All seeds (from 0 to 4) produced identical assignments relative to the baseline. This indicates stability across the tested random initializations.
It does not remove the weak separation indicated by the silhouette score.

Testing sensitivity to removing the coverage-awareness feature while keeping two clusters.

In [ ]:
features_reduced = df_features_prepared.drop(
    columns=[awareness_column]
)

X_reduced = pd.get_dummies(
    features_reduced,
    drop_first=False,
    dtype=int
)

reduced_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

reduced_labels = reduced_model.fit_predict(X_reduced)

print("Reduced matrix shape:", X_reduced.shape)
print(
    "Silhouette score:",
    round(silhouette_score(X_reduced, reduced_labels), 3)
)
print(
    "ARI compared with baseline:",
    round(adjusted_rand_score(baseline_labels, reduced_labels), 3)
)
print("\nCluster sizes:")
print(pd.Series(reduced_labels).value_counts().sort_index())

**Findings:** The ARI of 0.93 suggests a strong agreement with the baseline.

Reporting the baseline and reduced silhouette scores and their ARI agreement to six decimal places.

In [ ]:
print(
    "Baseline silhouette:",
    f"{silhouette_score(X, baseline_labels):.6f}"
)

print(
    "Reduced silhouette:",
    f"{silhouette_score(X_reduced, reduced_labels):.6f}"
)

print(
    "ARI versus baseline:",
    f"{adjusted_rand_score(baseline_labels, reduced_labels):.6f}"
)

**Findings:** The Baseline Silhouette is 0.121 and the reduced Silhouette is 0.133. Both suggest weak separation. The ARI is 0.930, which suggests that removing coverage awareness has limited influence on assignments.

The feature coverage awareness will be removed since the comparison has shown that removing the question largely preserves the grouping. It also simplifies the representation because it avoids the 133 missing answers.

Checking how removing coverage awareness affects the interpretation of the two groups.

In [ ]:
reduced_cluster_labels = pd.Series(
    reduced_labels,
    index=X_reduced.index,
    name="cluster"
)

reduced_profiles = X_reduced.groupby(reduced_cluster_labels).mean().T * 100

with pd.option_context("display.max_rows", None):
    display(reduced_profiles.round(1))

**Findings:** After removing the feature coverage awareness, cluster 1 reports greater perceived support and openness, while cluster 0 reports greater barriers and uncertainty. Within cluster percentages, 65.1% vs. 8.0% report equal seriousness, and 81.9% vs. 8.5% feel comfortable discussing mental health with supervisors. The separation remains weak with a Silhouette Score of around 0.133.

## Dimensionality Reduction and Visualization

The one-hot encoded responses to the nine survey questions will be projected into two dimensions and the respondents will be colored according to their existing K-means cluster assignment.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_reduced)

print("Projected shape:", X_pca.shape)
print("Variance explained by PC1 and PC2:",
      (pca.explained_variance_ratio_ * 100).round(1))
print(f"Total variance explained: {pca.explained_variance_ratio_.sum(): .1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for cluster in [0, 1]:
    mask = reduced_labels == cluster
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        s=18, alpha=0.45, label=f"Cluster {cluster}"
    )

ax.set(
    xlabel="PC1 (18.0% explained variance)",
    ylabel="PC2 (9.3% explained variance)",
    title="PCA projection of the two K-means clusters"
)
ax.legend()
plt.show()

**Finding:** The two clusters mainly differ along PC1. Cluster 0 (blue) lies mostly to the left and cluster 1 (orange) lies mostly to the right, while some overlap near the boundary. The two components capture 27.3% of the variance. The silhouette score of 0.133 indicates a weak separation in the full encoded feature space.

Creating a grouped bar chart to show what differs between the clusters. The percentage of respondents in each cluster who answered "Yes" will be compared to three workplace mental health questions.

In [ ]:
selected_responses = {
    f"{clustering_features[8]}_Yes": "Comfortable speaking to a supervisor",
    f"{clustering_features[9]}_Yes": "Mental and physical health taken equally seriously",
    f"{clustering_features[3]}_Yes": "Employer offers mental health resources",
}

chart_data = (
    reduced_profiles
    .loc[list(selected_responses), [0, 1]]
    .rename(index=selected_responses, columns={0: "Cluster 0", 1: "Cluster 1"})
)

ax = chart_data.plot.barh(
    figsize=(10, 4),
    color=["tab:blue", "tab:orange"]
)

ax.set(
    title="Workplace mental health responses by cluster",
    xlabel="Respondents answering Yes within each cluster (%)",
    xlim=(0, 100)
)

ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Findings:** Among the three questions shown, the largest gap was in comfort discussing mental health with a supervisor. Cluster 0: 8.5% vs. Cluster 1: 81.9%; so the difference is 73.4 percentage points.
Cluster 1 also had higher "Yes" response rates on the other two questions, though the gaps were a bit smaller.

## Practical Implications and Limitations

The key finding of this analysis is that 8.5% of respondents in Cluster 0 reported feeling comfortable discussing mental health with a direct supervisor, while 81.9% of respondents in Cluster 1 reported the same.
This leads to a suggestion, that HR should take action, for example, by piloting the training of supervisors on how they should respond to employees, when they are entrusted with mental health topics by these employees. Before this training, an anonymous survey should conducted. After completing these trainings, another anonymous survey should be undertaken, to see whether the reported comfort improves.
A Silhouette score of 0.133 has indicated that the two clusters are weakly separated. This survey described differences in reported experiences, but it doesn't explain what caused the gap.